# Delivery times of three apps as smooth density curves

**Kernel Density Estimation** · a worked example from the StatNex teaching guide

**The story.** The table holds the delivery times in minutes of 50 orders from each of three food-delivery apps: ZipEats, FeastNow and UrbanBite. A kernel density curve is a smoothed histogram that does not depend on where you put the class boundaries.

**The question.** What is the smooth shape of each app's delivery-time distribution?

**To run it:**

1. Choose **Copy to Drive** at the top (or **File → Save a copy in Drive**). Your copy opens in a new tab. It is yours, so Colab runs it without a warning, and it keeps your changes.
2. In your copy, choose **Runtime → Run all**.

You can also run this page without a copy. Colab then warns that the notebook was not written by Google. It reads none of your data, so **Run anyway** is safe, but your changes are not kept.

The first cells save the table in this session. The rest is the calculation StatNex runs, written out in Python. It does not use StatNex, and every number it prints was checked against the app's.

## Before the analysis

In [ ]:
# Each script saves its figures to the folder figures/, as it would on your computer.
# This shows each new figure under the cell that saved it.
from pathlib import Path

import matplotlib
from IPython import get_ipython
from IPython.display import Image, display

matplotlib.use("Agg")
shown = set()


def show_new_figures(*_):
    for path in sorted(Path("figures").glob("*.png"), key=lambda p: p.stat().st_mtime_ns):
        if (path, path.stat().st_mtime_ns) not in shown:
            shown.add((path, path.stat().st_mtime_ns))
            display(Image(filename=str(path)))


events = get_ipython().events
for old in [f for f in events.callbacks["post_run_cell"] if f.__name__ == "show_new_figures"]:
    events.unregister("post_run_cell", old)
events.register("post_run_cell", show_new_figures)

## The table

This cell saves the table `01_basic/kernel_density.csv` (50 rows) as `data.csv`, the file the code reads.

In [ ]:
%%writefile data.csv
No.,ZipEats,FeastNow,UrbanBite
1.0,16.0,32.0,36.0
2.0,24.0,26.0,41.0
3.0,28.0,31.0,38.0
4.0,19.0,33.0,48.0
5.0,29.0,25.0,33.0
6.0,26.0,37.0,42.0
7.0,22.0,30.0,42.0
8.0,30.0,28.0,42.0
9.0,22.0,29.0,34.0
10.0,26.0,35.0,46.0
11.0,28.0,33.0,38.0
12.0,17.0,27.0,53.0
13.0,29.0,34.0,41.0
14.0,26.0,26.0,45.0
15.0,23.0,23.0,35.0
16.0,23.0,30.0,46.0
17.0,23.0,30.0,47.0
18.0,22.0,33.0,33.0
19.0,26.0,35.0,44.0
20.0,12.0,29.0,31.0
21.0,26.0,29.0,32.0
22.0,14.0,42.0,44.0
23.0,29.0,33.0,42.0
24.0,20.0,32.0,39.0
25.0,23.0,32.0,41.0
26.0,20.0,41.0,38.0
27.0,27.0,29.0,46.0
28.0,23.0,36.0,38.0
29.0,28.0,34.0,36.0
30.0,19.0,33.0,31.0
31.0,27.0,30.0,36.0
32.0,27.0,32.0,38.0
33.0,29.0,38.0,32.0
34.0,24.0,35.0,46.0
35.0,26.0,41.0,45.0
36.0,18.0,31.0,38.0
37.0,25.0,37.0,44.0
38.0,58.0,37.0,34.0
39.0,17.0,33.0,32.0
40.0,31.0,31.0,29.0
41.0,27.0,40.0,45.0
42.0,18.0,39.0,40.0
43.0,24.0,35.0,37.0
44.0,29.0,37.0,42.0
45.0,25.0,35.0,43.0
46.0,24.0,35.0,36.0
47.0,25.0,31.0,39.0
48.0,23.0,33.0,40.0
49.0,20.0,37.0,46.0
50.0,26.0,33.0,44.0

## Libraries

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats

pd.set_option("display.width", 120)

## The data

In [ ]:
# One column per variable. Blank and text cells are left out, so columns may differ
# in length.
data = pd.read_csv("data.csv", keep_default_na=False, na_values=[""])
columns = ["ZipEats", "FeastNow", "UrbanBite"]
series = {c: pd.to_numeric(data[c], errors="coerce").dropna().to_numpy(float) for c in columns}

## Descriptive statistics

In [ ]:
# Variance and SD divide by n - 1. Quartiles interpolate linearly between the sorted
# values (NumPy's default, Excel's QUARTILE.INC).
def describe(x):
    q1, median, q3 = np.percentile(x, [25, 50, 75])
    return {"n": x.size, "mean": np.mean(x), "variance": np.var(x, ddof=1),
            "sd": np.std(x, ddof=1), "min": x.min(), "q1": q1, "median": median,
            "q3": q3, "max": x.max(), "iqr": q3 - q1}


summary = pd.DataFrame({name: describe(x) for name, x in series.items()})
print(summary)

## Kernel density estimate

In [ ]:
# A normal bump on every observation, added up, and divided by n so that the area
# under the curve is 1. Read it like a smooth histogram: a tall, narrow hill means
# values packed close together; a second hill, two groups; a small separate bump, a
# few values far from the rest.
# The bump's width (the bandwidth, h) is Silverman's rule of thumb, R's default:
# 0.9 × min(SD, IQR / 1.34) × n^(-1/5). The IQR keeps one extreme value from
# widening every hill. SciPy's own "silverman" option is a different formula
# (about 1.06 × SD × n^(-1/5)), so h is worked out here and handed to gaussian_kde.
def bandwidth(x):
    sd = np.std(x, ddof=1)
    q1, q3 = np.percentile(x, [25, 75])
    spread = min(sd, (q3 - q1) / 1.34) or sd
    return 0.9 * spread * x.size ** (-1 / 5)


# gaussian_kde scales its bump by the data's SD, so it is given h / SD. The curve is
# drawn on 100 points running three bandwidths beyond the smallest and
# largest value.
points = 100
bandwidths = {}
curves = {}
for name, x in series.items():
    h = bandwidth(x)
    kde = stats.gaussian_kde(x, bw_method=h / np.std(x, ddof=1))
    grid = np.linspace(x.min() - 3 * h, x.max() + 3 * h, points)
    bandwidths[name] = h
    curves[name] = (grid, kde(grid))
print(pd.Series(bandwidths, name="bandwidth h"))

Path("figures").mkdir(exist_ok=True)
fig, ax = plt.subplots()
for name, (grid, density) in curves.items():
    ax.plot(grid, density, label=name)
ax.set_xlabel("Value")
ax.set_ylabel("Density")
ax.legend()
fig.savefig("figures/kernel_density.png", dpi=150)
plt.close(fig)

## What the result says

A kernel density curve is a smoothed histogram. It puts a small hill on every order and adds the hills up. The bandwidth h is the width of each small hill: 1.54 minutes for ZipEats and FeastNow and 2.2 for UrbanBite. A larger h gives a smoother curve.

Each app has one main hill, and the hills come later from ZipEats (mean 24.46 minutes) to FeastNow (32.94) to UrbanBite (39.76). ZipEats' slowest order, 58 minutes, is far from the rest, so it shows as a tiny bump of its own.

The hills overlap at their edges. FeastNow's fastest order took 23 minutes, less than the ZipEats mean. So an app that is faster on average is not faster every time.

## Your own data

Open the **Files** panel (the folder icon on the left) and upload a CSV file. Then change `"data.csv"` in the `read_csv` line to its name, change the column names in the code to yours, and choose **Run all** again.